# The `profiling` package — full walkthrough

One package, two front ends. The Streamlit dashboard in `app.py` imports it, and so does
this notebook. Nothing in `profiling` knows about Streamlit, so **every** capability shown
below is equally available in a script, a notebook, a batch job or a Databricks job.

| Module | What lives there |
|---|---|
| `profiling.dataio` | loading, delimiter detection, lossless memory tuning, stacking frames, collapsing one-hot encodings |
| `profiling.stats` | descriptive statistics, missing values, correlation |
| `profiling.summaries` | pre-aggregated distribution and outlier summaries |
| `profiling.grouping` | splitting a dataset into segments |
| `profiling.plots` | interactive Plotly figures |
| `profiling.static` | static Matplotlib figures |
| `profiling.pipeline` | orchestration — one run, or one run per segment |
| `profiling.report` | PDF, ZIP and folder export |

The design rule that makes this work: **every chart, interactive or exported, is drawn from
the same pre-aggregated summary**, never from the raw column. A Plotly chart on screen and a
Matplotlib chart in the PDF therefore cannot disagree, and the payload per chart is a few
hundred bytes regardless of how many rows the file has.

---
## 1. Setup

In [ ]:
import json
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd

# Works whether Jupyter was started at the repository root or inside Examples/.
ROOT = Path.cwd()
if not (ROOT / "profiling").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA = ROOT / "Examples" / "Data"
OUT = Path(tempfile.mkdtemp(prefix="profiling_walkthrough_"))

import profiling

print("profiling", profiling.__version__)
print("data     ", DATA)
print("outputs  ", OUT)

Importing `profiling` costs pandas and NumPy and nothing else. Matplotlib and Plotly are
pulled in the first time a figure or export name is touched, so a script that only wants
numbers never pays for them.

In [ ]:
print("matplotlib imported?", "matplotlib" in sys.modules)
print("plotly imported?    ", "plotly" in sys.modules)

Figures are built through Matplotlib's object-oriented API — real `Figure` objects, never
`pyplot`. `pyplot` keeps a global reference to every figure it creates, which slowly leaks
memory in a long-running process. That means we display them explicitly rather than relying
on `%matplotlib inline`:

In [ ]:
from IPython.display import Image, display


def show(figure, dpi=110, crop=True):
    """Render a Matplotlib Figure inline without touching the pyplot registry."""
    display(Image(profiling.figure_to_png(figure, dpi=dpi, crop=crop)))

---
## 2. Loading a dataset

`load_table` does four things at once: detects the delimiter, parses with PyArrow's
multi-threaded reader (falling back to the pandas C parser), fingerprints the raw bytes, and
applies **lossless** dtype conversions. Only conversions that cannot change a value are
applied — integers are narrowed to the smallest exact type and repetitive text becomes
`category`. Floating-point precision is never reduced.

In [ ]:
train = profiling.load_table(DATA / "HousingData_TrainData.csv")

print(f"source     {train.source_name}")
print(f"shape      {train.n_rows:,} rows x {train.n_cols} columns")
print(f"parser     {train.parser}")
print(f"delimiter  {train.delimiter!r}")
print(f"fingerprint blake2b-128 {train.token}")
print(f"memory     {train.memory_before_mb:.2f} MB -> {train.memory_after_mb:.2f} MB "
      f"({train.memory_saved_mb:.2f} MB reclaimed)")
print(f"conversions {train.optimizations}")

df = train.frame
df.head()

The dtypes as they were in the file are kept, so a report can show the source types even after downcasting:

In [ ]:
pd.DataFrame(
    {"in_file": pd.Series(train.original_dtypes), "in_memory": df.dtypes.astype(str)}
)

In [ ]:
profiling.dtype_summary(df)

---
## 3. Descriptive statistics

Every quantity below is read off **one sorted copy** of each column: cardinality, mode,
zeros, min/max and all quantiles. That replaces the six separate passes (`value_counts`,
`nunique`, three `quantile` calls, a min/max scan) a naive implementation makes.

In [ ]:
profiling.describe_data(df)

`numeric_only=False` profiles text and categorical columns too; their numeric fields come back null:

In [ ]:
profiling.describe_data(df, numeric_only=False)

`round_columns` renders chosen columns as formatted strings — good for display, wrong for a
CSV export, which is why it is off by default. `drop_columns` trims the output.

In [ ]:
profiling.describe_data(
    df,
    round_columns={"Min": 2, "50%": 2, "Max": 2},
    drop_columns=["n_Zeros", "Perc. of Zeros", "1%", "99%"],
)

---
## 4. Missing values

A value is missing when pandas reports it as null (`NaN`, `NaT`, `None`). Percentages are
relative to the **total** row count, not the non-null count.

In [ ]:
profiling.missing_count(df)

In [ ]:
# non_zero_only=False lists every column, which is what an audit trail wants.
profiling.missing_count(df, non_zero_only=False)

---
## 5. Correlation

Pearson on raw values, Spearman as Pearson on average ranks, Kendall as tau-b with tie
correction. Missing values are handled pairwise-complete, matching `DataFrame.corr`. With no
gaps in the data Pearson and Spearman each reduce to a single BLAS matrix product; Kendall
cannot be reduced that way, so its pairs are spread over a thread pool.

A constant feature has an undefined coefficient and is left blank rather than reported as
zero.

In [ ]:
pearson = profiling.correlation_matrix(df, "pearson")
pearson.round(3)

`correlation_matrices` extracts the numeric matrix **once** and reuses it for every method,
which is why it beats calling `correlation_matrix` in a loop.

In [ ]:
matrices = profiling.correlation_matrices(df, ("pearson", "spearman", "kendall"))
for method, matrix in matrices.items():
    print(f"{method:<9} {matrix.shape}")
matrices["kendall"].round(3).iloc[:5, :5]

In [ ]:
profiling.top_absolute_correlations(matrices["spearman"], threshold=0.5, method_label="spearman")

On a large file the correlation step can be run on a **seeded** subsample. The draw applies
to every method, all computed from the same sampled rows, so the methods stay comparable —
and the seed is recorded, so the number is reproducible.

In [ ]:
sampled = profiling.correlation_matrix(df, "spearman", max_rows=2_000, random_state=7)
again = profiling.correlation_matrix(df, "spearman", max_rows=2_000, random_state=7)
print("reproducible:", sampled.equals(again))
(sampled - matrices["spearman"]).abs().max().max()

---
## 6. Distribution summaries

A summary, not a chart. `NumericDistribution` holds bin edges and counts;
`CategoricalDistribution` holds value counts with the long tail folded into `Other`. Both
are tiny and both are what every renderer consumes.

In [ ]:
numeric = profiling.numeric_distribution(df["MedInc"], binning=profiling.EQUAL_WIDTH, n_bins=8)
print("feature ", numeric.feature)
print("binning ", numeric.binning, f"({numeric.n_bins} bins)")
print("valid   ", f"{numeric.n_valid:,}  missing {numeric.n_missing:,}")
print("range   ", f"{numeric.minimum:.4g} .. {numeric.maximum:.4g}")
print("labels  ", numeric.labels)
print("counts  ", numeric.counts)
print("centers ", numeric.centers.round(3))

Quantile binning splits on empirical quantiles and drops duplicate edges. A column with no
more distinct values than requested bins gets one bin per distinct value, which avoids the
degenerate single-bar result on flag-like columns.

In [ ]:
quantile = profiling.numeric_distribution(df["MedInc"], binning=profiling.QUANTILE, n_bins=8)
print(quantile.labels)
print(quantile.counts, "<- near-equal occupancy, by construction")

flag = profiling.numeric_distribution(df["Target"], binning=profiling.QUANTILE, n_bins=8)
print("\nflag-like column ->", flag.labels, flag.counts)

In [ ]:
categorical = profiling.categorical_distribution(df["Education"])
print(categorical.categories, categorical.counts, "truncated:", categorical.truncated)

`build_distributions` dispatches per column and `distributions_to_frame` flattens the lot for export:

In [ ]:
distributions = profiling.build_distributions(df, binning=profiling.QUANTILE, n_bins=6)
print(len(distributions), "summaries:", list(distributions))
profiling.distributions_to_frame(distributions).head(10)

---
## 7. Outlier summaries

Five-number summary plus Tukey fences. The box spans Q1 to Q3 and reaches the most extreme
observation still within 1.5 x IQR of the nearer quartile. **Outlier counts are exact**; when
a feature has more outliers than can be drawn legibly an evenly spaced subset is kept for
plotting and the summary says so.

In [ ]:
box = profiling.box_summary(df["AveOccup"])
print(f"feature      {box.feature}")
print(f"n_valid      {box.n_valid:,}   n_missing {box.n_missing:,}")
print(f"min/max      {box.minimum:.4g} / {box.maximum:.4g}")
print(f"Q1/med/Q3    {box.q1:.4g} / {box.median:.4g} / {box.q3:.4g}   IQR {box.iqr:.4g}")
print(f"fences       {box.lower_fence:.4g} .. {box.upper_fence:.4g}")
print(f"outliers     {box.n_outliers:,} ({box.n_outliers_low:,} low, {box.n_outliers_high:,} high)")
print(f"drawn        {box.outliers.size:,}  truncated={box.outliers_truncated}")

In [ ]:
boxes = profiling.build_box_summaries(df)
profiling.box_summaries_to_frame(boxes).round(4)

---
## 8. Static figures (Matplotlib)

These are the figures the PDF and ZIP exports are made of. They consume the summaries
computed above, so nothing is recomputed to draw them.

In [ ]:
show(profiling.histogram_figure(distributions["MedInc"], show_percentage=True))

In [ ]:
show(profiling.histogram_figure(distributions["Education"], color=profiling.PALETTE[2]))

In [ ]:
show(profiling.box_figure(boxes["AveOccup"], color=profiling.PALETTE[1]))

In [ ]:
show(profiling.heatmap_figure(matrices["spearman"], "spearman", threshold=0.6, fit_page=False))

In [ ]:
show(profiling.missing_bar_figure(profiling.missing_count(df), fit_page=False))

`dataframe_figure` renders any table as a single image — for a slide, or a picture in a report:

In [ ]:
show(
    profiling.dataframe_figure(
        profiling.describe_data(df).head(6),
        title="Descriptive statistics (first 6 features)",
        show_index=True,
    ),
    dpi=100,
)

The grid builders paginate onto the shared A4 landscape page used by the PDF:

In [ ]:
pages = profiling.histogram_grid_figures(list(distributions.values()), n_cols=3, n_rows=3)
print(len(pages), "page(s)")
show(pages[0], dpi=90, crop=False)

In [ ]:
show(profiling.box_grid_figures(list(boxes.values()), n_rows=6)[0], dpi=90, crop=False)

Any figure can go straight to disk. `save_figure` and `figure_to_png` both bypass `pyplot` entirely:

In [ ]:
target = OUT / "medinc_histogram.png"
profiling.save_figure(profiling.histogram_figure(distributions["MedInc"]), str(target), dpi=150)
print(target.name, f"{target.stat().st_size:,} bytes")

---
## 9. Interactive figures (Plotly)

The same summaries, rendered for the browser. This is exactly what the dashboard shows.

In [ ]:
profiling.plots.histogram_figure(distributions["MedInc"], show_percentage=True)

In [ ]:
profiling.plots.box_figure(boxes["AveOccup"])

In [ ]:
profiling.plots.correlation_heatmap(matrices["spearman"], "spearman", threshold=0.6)

---
## 10. The whole thing in one call

`ProfilingSettings` captures everything that influences the numbers, and the entire object is
written into the exported metadata — so a report can always be traced back to the
configuration that produced it.

In [ ]:
settings = profiling.ProfilingSettings(
    include_describe=True,
    include_missing=True,
    include_correlation=True,
    include_histograms=True,
    include_box_plots=True,
    correlation_methods=("pearson", "spearman"),
    correlation_threshold=0.6,
    binning=profiling.QUANTILE,
    n_bins=8,
    show_percentage=True,
    random_state=0,
)
settings

In [ ]:
result = profiling.run_profiling(
    df,
    settings,
    dataset_name="HousingData_TrainData.csv",
    dtype_labels=train.original_dtypes,
    source_info={
        "parser": train.parser,
        "delimiter": train.delimiter,
        "token": train.token,
        "memory_saved_mb": train.memory_saved_mb,
    },
    progress=lambda label, fraction: print(f"  {fraction:5.0%}  {label}"),
)

print()
print(f"rows            {result.n_rows:,}")
print(f"numeric features {len(result.numeric_features)}")
print(f"in-memory        {result.memory_mb:.2f} MB")
print(f"total            {result.total_seconds:.3f} s")
print("timings         ", {k: round(v, 4) for k, v in result.timings.items()})
print("notes           ", result.notes)

In [ ]:
result.describe

In [ ]:
result.top_correlations["spearman"]

The run is self-describing. `metadata()` is JSON-serialisable and records the dataset, the settings, the timings and the library versions:

In [ ]:
print(json.dumps(result.metadata(), indent=2, default=str)[:1800], "...")

---
## 11. Exporting

Three shapes of the same bundle: a paginated PDF, a ZIP archive, or a plain folder.
`build_zip` and `write_report_dir` share one definition of the contents (`bundle_files`), so
an unpacked archive and a written folder are the same tree.

In [ ]:
pdf = profiling.build_pdf(result)
(OUT / "report.pdf").write_bytes(pdf)
print(f"PDF {len(pdf):,} bytes")

# Pass the rendered PDF in so the ZIP does not have to re-render it.
archive = profiling.build_zip(result, pdf_bytes=pdf)
(OUT / f"{profiling.suggested_basename(result)}.zip").write_bytes(archive)
print(f"ZIP {len(archive):,} bytes")

In [ ]:
written = profiling.write_report_dir(result, OUT / "report_folder", include_pdf=False)
for path in written:
    print(Path(path).relative_to(OUT / "report_folder"))

`bundle_files` is the underlying stream, if you want to route the contents somewhere else — object storage, a database, an email attachment:

In [ ]:
for name, payload in profiling.bundle_files(result, include_pdf=False, include_individual_charts=False):
    print(f"{len(payload):>9,}  {name}")

---
## 12. Preparation: collapsing one-hot encodings

Profiling a wide block of 0/1 dummies says very little; profiling the one categorical column
they encode says a lot. Each dummy group is replaced, in the position of its first member, by
a single column.

In [ ]:
ohe = profiling.load_table(DATA / "HousingData_OHE_TrainData.csv").frame
print(list(ohe.columns))
ohe[["Education_A", "Education_B", "Education_C"]].head()

In [ ]:
# "auto" infers the groups from the separator in the column names.
merged = profiling.merge_one_hot_encoded_columns(ohe, separator="_")
print(list(merged.columns))
merged["Education"].value_counts()

In [ ]:
# strip_prefix drops the "<feature><separator>" part of the value.
stripped = profiling.merge_one_hot_encoded_columns(
    ohe, {"Education": ["Education_A", "Education_B", "Education_C"]},
    separator="_", strip_prefix=True,
)
stripped["Education"].value_counts()

In [ ]:
# A list of original feature names groups by substring match instead.
profiling.merge_one_hot_encoded_columns(ohe, ["Education"], separator="_").head(3)

The input frame is never modified — a copy comes back. Note that a row in which every dummy
is zero is attributed to the first column of its group, because the winner is simply the
largest dummy.

In [ ]:
print("original untouched:", list(ohe.columns))

---
## 13. Preparation: stacking several frames

`combine_datasets` stacks a mapping into one frame with a column recording where each row
came from. That column is excluded from `feature_names`, so it is never profiled as a
feature — and it is exactly what you then group on to compare train against test.

In [ ]:
test = profiling.load_table(DATA / "HousingData_TestData.csv")

combined = profiling.combine_datasets({"Train": train.frame, "Test": test.frame})
print("shape        ", combined.frame.shape)
print("tag column   ", combined.tag_column)
print("sources      ", combined.sources)
print("features     ", combined.feature_names)
combined.frame["Dataset"].value_counts()

A single frame passed in comes back untouched, with `tag_column=None`. That lets a caller route its input through this function without special-casing:

In [ ]:
passthrough = profiling.combine_datasets(df)
print("is_combined:", passthrough.is_combined, "| tag_column:", passthrough.tag_column)
print("same object:", passthrough.frame is df)

---
## 14. Segmentation

This is the capability the dashboard does not expose. A dataset can be split four ways, and
every analysis is then run per segment.

| Call | Split |
|---|---|
| `make_grouping(df, "Education")` | one segment per distinct value |
| `make_grouping(df, "HouseAge", bins=[(1, 20), (20, 50)])` | explicit right-closed intervals |
| `make_grouping(df, "HouseAge", n_bins=4)` | four equal-width intervals |
| `make_grouping(df, "HouseAge", n_quantiles=4)` | four equal-frequency intervals |

A `Grouping` holds only row indices, never data, so building one is cheap and the same
grouping can be reused for several analyses.

In [ ]:
by_value = profiling.make_grouping(df, "Education")
print(by_value.method, "|", by_value.grouping_var, "|", by_value.labels)
by_value.to_frame()

In [ ]:
by_interval = profiling.make_grouping(df, "HouseAge", bins=[(1, 20), (20, 50)])
print(by_interval.labels)
print(f"{by_interval.n_rows_unassigned:,} rows fall outside every interval and belong to no segment")
by_interval.to_frame()

In [ ]:
# Equal-width: even intervals, uneven occupancy. A bin can even come out empty.
profiling.make_grouping(df, "HouseAge", n_bins=4).to_frame()

In [ ]:
# Quantiles: uneven intervals, even occupancy.
by_quantile = profiling.make_grouping(df, "HouseAge", n_quantiles=4)
by_quantile.to_frame()

`by` also accepts a Series or a bare array, so a segment can be defined by something that is not a column:

In [ ]:
derived = (df["MedInc"] > df["MedInc"].median()).rename("AboveMedianIncome")
profiling.make_grouping(df, derived).to_frame()

Iterating a grouping slices the frame lazily, one segment at a time:

In [ ]:
for segment, subset in by_quantile.frames(df):
    print(f"{segment.label:<28} {segment.n_rows:>6,} rows   "
          f"mean MedInc {subset['MedInc'].mean():.3f}")

In [ ]:
# Indexing by key or by position, and the guard against ambiguous requests.
print(by_quantile[0].label)
print(by_quantile[by_quantile.keys[-1]].n_rows)
try:
    profiling.make_grouping(df, "HouseAge", n_bins=4, n_quantiles=4)
except ValueError as exc:
    print("guarded:", exc)

---
## 15. Profiling every segment

`run_segmented_profiling` applies the identical settings to every segment — that is what
keeps the numbers comparable — and returns one ordinary `ProfilingResult` per segment. So
anything that renders or exports a single run also handles one segment of a segmented run,
with no second code path.

In [ ]:
segmented = profiling.run_segmented_profiling(
    df,
    settings,
    by="HouseAge",
    n_quantiles=4,
    dataset_name="HousingData_TrainData.csv",
)

print(f"grouping   {segmented.grouping_var} ({segmented.method})")
print(f"segments   {len(segmented)}")
print(f"total      {segmented.total_seconds:.3f} s")
print(f"notes      {segmented.notes}")
segmented.segment_overview()

In [ ]:
# Each segment is a plain ProfilingResult.
one = segmented[segmented.keys[0]]
print(type(one).__name__, "|", one.dataset_name)
one.describe.head()

The comparison methods return long-format frames with a `segment` column in front — the
shape that pivots, plots and diffs cleanly.

In [ ]:
comparison = segmented.describe_comparison()
print(comparison.shape)
comparison.head(12)

In [ ]:
# The median of every feature, segment by segment.
comparison.pivot(index="feature", columns="segment", values="50%").round(3)

In [ ]:
segmented.missing_comparison(affected_only=True)

In [ ]:
segmented.box_comparison()[
    ["segment", "feature", "median", "iqr", "n_outliers", "perc_outliers"]
].head(12)

In [ ]:
segmented.distribution_comparison().head(10)

`top_correlation_comparison` stacks the above-threshold pairs per segment. A pair listed for
one segment and not another is the interesting case: it cleared the threshold there and not
here.

In [ ]:
segmented.top_correlation_comparison("spearman")

`correlation_comparison` is built from the **full** matrices instead, so every pair is
present in every segment and the row can be read across. Sorted by spread, the top rows are
the relationships that appear, vanish or flip sign between segments.

In [ ]:
drift = segmented.correlation_comparison("pearson")
drift.head(8).round(3)

In [ ]:
# Same thing, restricted to pairs that are actually strong somewhere.
segmented.correlation_comparison("spearman", min_abs=0.5).round(3)

When segmenting on distinct values the grouping column is constant inside each segment, so
its own profile is degenerate. The run says so in its notes, and
`include_grouping_column=False` leaves it out.

In [ ]:
kept = profiling.run_segmented_profiling(df, settings, by="Education")
print("notes:", kept.notes)

dropped = profiling.run_segmented_profiling(
    df, settings, by="Education", include_grouping_column=False
)
print("with column   ", len(kept.settings.selected_features), "features")
print("without column", len(dropped.settings.selected_features), "features")

With `by=None` the whole dataset becomes the single segment `Data=All`, so a caller can always take the segmented path:

In [ ]:
ungrouped = profiling.run_segmented_profiling(df, settings)
print(ungrouped.keys, "|", ungrouped.grouping.method)

---
## 16. Train vs test drift

Combine the two frames, then segment on the tag column. Two lines, and every comparison table
above becomes a drift report.

In [ ]:
drift_run = profiling.run_segmented_profiling(
    {"Train": train.frame, "Test": test.frame},
    settings,
    by="Dataset",
    dataset_name="HousingData",
)
print(drift_run.keys)
drift_run.segment_overview()

In [ ]:
medians = drift_run.describe_comparison().pivot(index="feature", columns="segment", values="50%")
medians["shift_%"] = ((medians["Test"] - medians["Train"]) / medians["Train"] * 100).round(2)
medians.round(4)

In [ ]:
outliers = drift_run.box_comparison().pivot(
    index="feature", columns="segment", values="perc_outliers"
)
outliers.round(3)

In [ ]:
drift_run.correlation_comparison("pearson").head(6).round(3)

---
## 17. Exporting a segmented run

One full bundle folder per segment, plus a `00_comparison/` folder holding the tables that
span every segment. The grouping is visible from the file tree alone.

In [ ]:
tree = profiling.write_segmented_report_dir(
    segmented, OUT / "segmented", include_pdf=False, include_individual_charts=False
)

for key, paths in tree.items():
    print(f"{key:<28} {len(paths):>3} files")

In [ ]:
root = OUT / "segmented"
for folder in sorted(p for p in root.iterdir() if p.is_dir()):
    print(folder.name + "/")
    for item in sorted(folder.rglob("*")):
        if item.is_file():
            print("   ", item.relative_to(folder).as_posix())

In [ ]:
pd.read_csv(root / "00_comparison" / "segment_overview.csv")

---
## 18. A wider dataset

Thirty numeric features, so the Kendall path and the pair ranking get a real workout. Kendall
is the one correlation that cannot be reduced to a matrix product — every pair needs its own
O(n log n) inversion count — so its 435 pairs are spread over a thread pool.

In [ ]:
cancer = profiling.load_table(DATA / "BreastCancer_TrainData.csv")
print(f"{cancer.n_rows:,} rows x {cancer.n_cols} columns")

wide = profiling.run_profiling(
    cancer.frame,
    profiling.ProfilingSettings(
        correlation_methods=("kendall",),
        correlation_threshold=0.8,
        binning=profiling.EQUAL_WIDTH,
        n_bins=20,
    ),
    dataset_name=cancer.source_name,
)
print(f"{wide.total_seconds:.2f} s   ", {k: round(v, 3) for k, v in wide.timings.items()})
wide.top_correlations["kendall"].head(10).round(4)

In [ ]:
show(
    profiling.heatmap_figure(wide.correlations["kendall"], "kendall", threshold=0.8, fit_page=False),
    dpi=80,
)

Segmenting a wide dataset by the label column is the standard class-imbalance and separability check:

In [ ]:
by_target = profiling.run_segmented_profiling(
    cancer.frame,
    profiling.ProfilingSettings(
        include_correlation=False, include_histograms=False, correlation_methods=()
    ),
    by="Target",
    include_grouping_column=False,
    dataset_name=cancer.source_name,
)
by_target.segment_overview()

In [ ]:
means = by_target.box_comparison().pivot(index="feature", columns="segment", values="mean")
means["ratio"] = (means["1"] / means["0"]).round(3)
means.sort_values("ratio").round(4).head(10)

---
## 19. Spark and Databricks

The package has **no** Spark dependency and nothing Spark-related is imported until a Spark
object is actually passed in. When one is:

* `make_grouping` bucketises **in Spark** — `Bucketizer` for `bins`/`n_bins` and
  `QuantileDiscretizer` for `n_quantiles` — instead of pulling the grouping column onto the
  driver;
* `combine_datasets` concatenates with `pyspark.pandas`;
* `run_segmented_profiling` materialises one segment at a time, so the driver never holds
  more than a single segment.

So the same call works on a Databricks cluster:

```python
segmented = profiling.run_segmented_profiling(
    spark_df, settings, by="age", n_quantiles=4
)
profiling.write_segmented_report_dir(segmented, "/dbfs/tmp/profiling")
```

`profiling.to_pandas(x)` materialises a Spark frame and passes anything else through
unchanged, so a helper of your own can normalise its input in one line without importing
anything Spark-related.

---
## 20. Cleanup

In [ ]:
import shutil

print("removing", OUT)
shutil.rmtree(OUT, ignore_errors=True)

---
## Cheat sheet

```python
import profiling

# load
loaded = profiling.load_table("data.csv")            # -> LoadedData

# prepare
frame = profiling.merge_one_hot_encoded_columns(loaded.frame, separator="__")
both  = profiling.combine_datasets({"Train": tr, "Test": te})   # -> CombinedData

# analyse, piece by piece
profiling.describe_data(frame)
profiling.missing_count(frame)
profiling.correlation_matrices(frame, ("pearson", "kendall"))
profiling.build_distributions(frame, binning=profiling.QUANTILE, n_bins=8)
profiling.build_box_summaries(frame)

# ...or in one call
settings = profiling.ProfilingSettings(correlation_methods=("spearman",))
result   = profiling.run_profiling(frame, settings, dataset_name="data.csv")

# ...or once per segment
segmented = profiling.run_segmented_profiling(frame, settings, by="age", n_quantiles=4)
segmented.describe_comparison()
segmented.correlation_comparison("spearman")

# export
profiling.build_pdf(result)                                  # -> bytes
profiling.build_zip(result)                                  # -> bytes
profiling.write_report_dir(result, "out/")                    # -> [paths]
profiling.write_segmented_report_dir(segmented, "out/")       # -> {segment: [paths]}
```